# Qasper dense_recency_heavy Standalone

Dense retrieval plus strongest evidence placed at the end of the prompt.

Notebook n?y t? ch?a to?n b? code ?? ch?y tr?n Kaggle/Colab. Kh?ng clone repo, kh?ng import t? `src/`. M?c ??nh ch? ch?y paper c? `MIN_DOC_WORDS >= 3000` ?? t?p trung v?o long-context.

In [ ]:
# Kaggle/Colab setup. Run once per fresh session.
# If you already hit a NumPy/SciPy import error, restart the notebook session before rerunning from the top.
!pip -q install --no-cache-dir --force-reinstall "numpy==1.26.4" "scipy==1.13.1" "scikit-learn==1.5.2"
!pip -q install --no-cache-dir "datasets>=2.19.0,<4.0.0" "pyarrow>=15.0.0" "sentence-transformers==3.0.1" "transformers==4.44.2" "tqdm>=4.66.0"


In [ ]:
VARIANT = "dense_recency_heavy"
SPLIT = "validation"
MIN_DOC_WORDS = 3000
LIMIT = None  # Set to 10 for a smoke test.
TOP_K = 5
RETRIEVE_K = 20
CHUNK_SIZE = 180
OVERLAP = 40
RETRIEVER_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GENERATOR_MODEL = "google/flan-t5-base"
OUTPUT_DIR = "outputs/independent"

CONFIG = {
    "variant": VARIANT,
    "split": SPLIT,
    "min_doc_words": MIN_DOC_WORDS,
    "limit": LIMIT,
    "top_k": TOP_K,
    "retrieve_k": RETRIEVE_K,
    "chunk_size": CHUNK_SIZE,
    "overlap": OVERLAP,
    "retriever_model": RETRIEVER_MODEL,
    "generator_model": GENERATOR_MODEL,
}
CONFIG

In [ ]:
from __future__ import annotations

import json
import math
import re
import time
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from statistics import mean, median
from typing import Any, Iterable

import numpy as np
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

In [ ]:
QASPER_REVISION = "cc58ffb39db7ff6ce1951e28e029996bf499304e"
QASPER_BASE_URL = f"https://huggingface.co/datasets/allenai/qasper/resolve/{QASPER_REVISION}/qasper"
QASPER_PARQUET_FILES = {
    "train": f"{QASPER_BASE_URL}/qasper-train.parquet",
    "validation": f"{QASPER_BASE_URL}/qasper-validation.parquet",
    "test": f"{QASPER_BASE_URL}/qasper-test.parquet",
}


def load_qasper(split: str = "validation"):
    if split not in QASPER_PARQUET_FILES:
        raise ValueError(f"Unknown split: {split}")
    return load_dataset("parquet", data_files={split: QASPER_PARQUET_FILES[split]}, split=split)


@dataclass(frozen=True)
class Chunk:
    chunk_id: str
    doc_id: str
    title: str
    section: str
    text: str


@dataclass(frozen=True)
class QAExample:
    doc_id: str
    question_id: str
    title: str
    question: str
    gold_answers: list[str]
    evidence: list[str]


def document_text(record: dict[str, Any]) -> str:
    parts = []
    abstract = str(record.get("abstract", "")).strip()
    if abstract:
        parts.append(abstract)

    full_text = record.get("full_text", {})
    sections = full_text.get("section_name", [])
    paragraphs_by_section = full_text.get("paragraphs", [])
    for section, paragraphs in zip(sections, paragraphs_by_section):
        section_parts = [str(section).strip()] if str(section).strip() else []
        section_parts.extend(str(paragraph).strip() for paragraph in paragraphs if str(paragraph).strip())
        if section_parts:
            parts.append("\n".join(section_parts))
    return "\n\n".join(parts)


def document_word_count(record: dict[str, Any]) -> int:
    return len(document_text(record).split())


def iter_answer_records(answers: Any) -> list[dict[str, Any]]:
    if isinstance(answers, list):
        return [answer for answer in answers if isinstance(answer, dict)]
    if not isinstance(answers, dict):
        return []

    answer_values = answers.get("answer", [])
    annotation_ids = answers.get("annotation_id", [])
    worker_ids = answers.get("worker_id", [])

    if isinstance(answer_values, dict):
        answer_values = [answer_values]
    if not isinstance(answer_values, list):
        return []

    records = []
    for index, answer_value in enumerate(answer_values):
        record = {"answer": answer_value}
        if isinstance(annotation_ids, list) and index < len(annotation_ids):
            record["annotation_id"] = annotation_ids[index]
        if isinstance(worker_ids, list) and index < len(worker_ids):
            record["worker_id"] = worker_ids[index]
        records.append(record)
    return records


def normalise_answer(answer: dict[str, Any]) -> str | None:
    data = answer.get("answer", answer)
    if data.get("unanswerable"):
        return "Unanswerable"
    if data.get("free_form_answer"):
        return str(data["free_form_answer"]).strip()
    if data.get("extractive_spans"):
        spans = [str(span).strip() for span in data["extractive_spans"] if str(span).strip()]
        if spans:
            return " ; ".join(spans)
    yes_no = data.get("yes_no")
    if yes_no is not None:
        return str(yes_no)
    return None


def normalise_evidence(answer: dict[str, Any]) -> list[str]:
    data = answer.get("answer", answer)
    evidence = data.get("evidence", answer.get("evidence", []))
    if not evidence:
        return []
    return [str(item).strip() for item in evidence if str(item).strip()]


def extract_qa_examples(record: dict[str, Any]) -> list[QAExample]:
    qas = record.get("qas", {})
    questions = qas.get("question", [])
    question_ids = qas.get("question_id", [])
    answers_list = qas.get("answers", [])

    examples: list[QAExample] = []
    for question, question_id, answers in zip(questions, question_ids, answers_list):
        gold_answers = []
        evidence = []
        for answer in iter_answer_records(answers):
            normalised = normalise_answer(answer)
            if normalised:
                gold_answers.append(normalised)
            evidence.extend(normalise_evidence(answer))
        examples.append(
            QAExample(
                doc_id=record["id"],
                question_id=question_id,
                title=record.get("title", ""),
                question=question,
                gold_answers=gold_answers,
                evidence=evidence,
            )
        )
    return examples


def chunk_words(text: str, *, chunk_size: int = 180, overlap: int = 40) -> list[str]:
    words = text.split()
    if not words:
        return []
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk_size")

    chunks: list[str] = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        window = words[start : start + chunk_size]
        if window:
            chunks.append(" ".join(window))
        if start + chunk_size >= len(words):
            break
    return chunks


def build_document_chunks(record: dict[str, Any], *, chunk_size: int = 180, overlap: int = 40) -> list[Chunk]:
    full_text = record.get("full_text", {})
    sections = full_text.get("section_name", [])
    paragraphs_by_section = full_text.get("paragraphs", [])
    chunks: list[Chunk] = []
    chunk_index = 0

    abstract = record.get("abstract", "")
    for text in chunk_words(abstract, chunk_size=chunk_size, overlap=overlap):
        chunks.append(Chunk(f"{record['id']}::abstract::{chunk_index}", record["id"], record.get("title", ""), "abstract", text))
        chunk_index += 1

    for section, paragraphs in zip(sections, paragraphs_by_section):
        section_text = " ".join(str(paragraph) for paragraph in paragraphs if str(paragraph).strip())
        for text in chunk_words(section_text, chunk_size=chunk_size, overlap=overlap):
            chunks.append(Chunk(f"{record['id']}::{chunk_index}", record["id"], record.get("title", ""), str(section), text))
            chunk_index += 1
    return chunks

In [ ]:
def normalize_text(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


def token_f1(prediction: str, gold: str) -> float:
    prediction_tokens = normalize_text(prediction)
    gold_tokens = normalize_text(gold)
    if not prediction_tokens or not gold_tokens:
        return 0.0
    overlap = Counter(prediction_tokens) & Counter(gold_tokens)
    overlap_count = sum(overlap.values())
    if overlap_count == 0:
        return 0.0
    precision = overlap_count / len(prediction_tokens)
    recall = overlap_count / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def best_f1(prediction: str, gold_answers: list[str]) -> float:
    return max([token_f1(prediction, gold) for gold in gold_answers], default=0.0)


def answer_string_recall(contexts: list[Chunk], gold_answers: list[str]) -> float:
    if not gold_answers:
        return 0.0
    joined_context = " ".join(chunk.text.lower() for chunk in contexts)
    hits = 0
    for answer in gold_answers:
        answer_text = answer.lower().strip()
        if answer_text and answer_text in joined_context:
            hits += 1
    return hits / len(gold_answers)


def context_recall(contexts: list[Chunk], gold_answers: list[str], evidence: list[str], threshold: float = 0.45) -> float:
    targets = evidence or gold_answers
    if not targets:
        return 0.0
    hits = 0
    for target in targets:
        best = max([token_f1(chunk.text, target) for chunk in contexts], default=0.0)
        if best >= threshold:
            hits += 1
    return hits / len(targets)


def context_precision(contexts: list[Chunk], gold_answers: list[str], evidence: list[str], threshold: float = 0.25) -> float:
    if not contexts:
        return 0.0
    targets = evidence or gold_answers
    if not targets:
        return 0.0
    relevant = 0
    for chunk in contexts:
        best = max([token_f1(chunk.text, target) for target in targets], default=0.0)
        if best >= threshold:
            relevant += 1
    return relevant / len(contexts)


def faithfulness(prediction: str, contexts: list[Chunk], threshold: float = 0.35) -> float:
    if prediction.strip().lower() == "unanswerable":
        return 1.0
    return 1.0 if max([token_f1(prediction, chunk.text) for chunk in contexts], default=0.0) >= threshold else 0.0


def answer_relevancy(prediction: str, question: str, gold_answers: list[str]) -> float:
    if prediction.strip().lower() == "unanswerable":
        return 0.0
    question_similarity = token_f1(prediction, question)
    answer_similarity = best_f1(prediction, gold_answers)
    return 0.3 * question_similarity + 0.7 * answer_similarity

In [ ]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


class DenseRetriever:
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self.chunks: list[Chunk] = []
        self.embeddings: np.ndarray | None = None

    def index(self, chunks: list[Chunk]) -> None:
        self.chunks = chunks
        if not chunks:
            self.embeddings = None
            return
        self.embeddings = self.model.encode([chunk.text for chunk in chunks], normalize_embeddings=True, show_progress_bar=False)

    def search(self, query: str, *, top_k: int = 5) -> list[tuple[Chunk, float]]:
        if self.embeddings is None or not self.chunks:
            return []
        query_embedding = self.model.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]
        scores = np.matmul(self.embeddings, query_embedding)
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(self.chunks[index], float(scores[index])) for index in top_indices]


class BM25Retriever:
    def __init__(self, *, k1: float = 1.5, b: float = 0.75) -> None:
        self.k1 = k1
        self.b = b
        self.chunks: list[Chunk] = []
        self.doc_freqs: Counter[str] = Counter()
        self.term_freqs: list[Counter[str]] = []
        self.doc_lengths: list[int] = []
        self.avg_doc_length = 0.0

    def index(self, chunks: list[Chunk]) -> None:
        self.chunks = chunks
        self.term_freqs = []
        self.doc_freqs = Counter()
        self.doc_lengths = []
        for chunk in chunks:
            terms = tokenize(chunk.text)
            term_freq = Counter(terms)
            self.term_freqs.append(term_freq)
            self.doc_lengths.append(len(terms))
            self.doc_freqs.update(term_freq.keys())
        self.avg_doc_length = sum(self.doc_lengths) / len(self.doc_lengths) if self.doc_lengths else 0.0

    def search(self, query: str, *, top_k: int = 5) -> list[tuple[Chunk, float]]:
        if not self.chunks:
            return []
        query_terms = tokenize(query)
        scores = []
        total_docs = len(self.chunks)
        for index, term_freq in enumerate(self.term_freqs):
            score = 0.0
            doc_length = self.doc_lengths[index]
            for term in query_terms:
                if term not in term_freq:
                    continue
                doc_freq = self.doc_freqs[term]
                idf = math.log(1 + (total_docs - doc_freq + 0.5) / (doc_freq + 0.5))
                frequency = term_freq[term]
                denominator = frequency + self.k1 * (1 - self.b + self.b * doc_length / max(self.avg_doc_length, 1e-9))
                score += idf * frequency * (self.k1 + 1) / denominator
            scores.append(score)
        ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
        return [(self.chunks[index], float(scores[index])) for index in ranked_indices if scores[index] > 0]


def reciprocal_rank_fusion(ranked_lists: Iterable[list[tuple[Chunk, float]]], *, top_k: int = 5, rrf_k: int = 60) -> list[tuple[Chunk, float]]:
    scores: dict[str, float] = defaultdict(float)
    chunks_by_id: dict[str, Chunk] = {}
    for ranked_list in ranked_lists:
        for rank, (chunk, _score) in enumerate(ranked_list, start=1):
            scores[chunk.chunk_id] += 1 / (rrf_k + rank)
            chunks_by_id[chunk.chunk_id] = chunk
    ranked_ids = sorted(scores, key=scores.get, reverse=True)[:top_k]
    return [(chunks_by_id[chunk_id], scores[chunk_id]) for chunk_id in ranked_ids]


def u_shaped_reorder(chunks: list[Chunk]) -> list[Chunk]:
    reordered: list[Chunk | None] = [None] * len(chunks)
    left = 0
    right = len(chunks) - 1
    for index, chunk in enumerate(chunks):
        if index % 2 == 0:
            reordered[left] = chunk
            left += 1
        else:
            reordered[right] = chunk
            right -= 1
    return [chunk for chunk in reordered if chunk is not None]


def recency_heavy_reorder(chunks: list[Chunk]) -> list[Chunk]:
    return list(reversed(chunks)) if len(chunks) > 1 else chunks


class SmallSeq2SeqGenerator:
    def __init__(self, model_name: str = "google/flan-t5-base"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)

    def answer(self, question: str, contexts: list[Chunk], *, max_input_tokens: int = 1024, max_new_tokens: int = 96) -> str:
        context_text = "\n\n".join(
            f"[{index + 1}] Title: {chunk.title}\nSection: {chunk.section}\n{chunk.text}"
            for index, chunk in enumerate(contexts)
        )
        prompt = (
            "Answer the question using only the provided context. "
            "If the answer is not in the context, answer Unanswerable.\n\n"
            f"Context:\n{context_text}\n\nQuestion: {question}\nAnswer:"
        )
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_input_tokens).to(self.device)
        outputs = self.model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=2)
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

In [ ]:
class BaseDensePipeline:
    def __init__(self, *, retriever_model: str, generator_model: str, chunk_size: int, overlap: int, top_k: int) -> None:
        self.retriever = DenseRetriever(retriever_model)
        self.generator = SmallSeq2SeqGenerator(generator_model)
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.top_k = top_k

    def index_document(self, record: dict[str, Any]) -> None:
        self.retriever.index(build_document_chunks(record, chunk_size=self.chunk_size, overlap=self.overlap))

    def answer(self, question: str) -> dict[str, Any]:
        retrieved = self.retriever.search(question, top_k=self.top_k)
        contexts = [chunk for chunk, _score in retrieved]
        return {"answer": self.generator.answer(question, contexts), "contexts": contexts, "scores": [score for _chunk, score in retrieved]}


class BM25OnlyPipeline:
    def __init__(self, *, generator_model: str, chunk_size: int, overlap: int, top_k: int) -> None:
        self.retriever = BM25Retriever()
        self.generator = SmallSeq2SeqGenerator(generator_model)
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.top_k = top_k

    def index_document(self, record: dict[str, Any]) -> None:
        self.retriever.index(build_document_chunks(record, chunk_size=self.chunk_size, overlap=self.overlap))

    def answer(self, question: str) -> dict[str, Any]:
        retrieved = self.retriever.search(question, top_k=self.top_k)
        contexts = [chunk for chunk, _score in retrieved]
        return {"answer": self.generator.answer(question, contexts), "contexts": contexts, "scores": [score for _chunk, score in retrieved]}


class DenseReorderPipeline(BaseDensePipeline):
    def __init__(self, *, reorder_mode: str, retriever_model: str, generator_model: str, chunk_size: int, overlap: int, top_k: int) -> None:
        super().__init__(retriever_model=retriever_model, generator_model=generator_model, chunk_size=chunk_size, overlap=overlap, top_k=top_k)
        self.reorder_mode = reorder_mode

    def answer(self, question: str) -> dict[str, Any]:
        retrieved = self.retriever.search(question, top_k=self.top_k)
        score_by_id = {chunk.chunk_id: score for chunk, score in retrieved}
        contexts = [chunk for chunk, _score in retrieved]
        if self.reorder_mode == "u_shape":
            contexts = u_shaped_reorder(contexts)
        elif self.reorder_mode == "recency_heavy":
            contexts = recency_heavy_reorder(contexts)
        return {"answer": self.generator.answer(question, contexts), "contexts": contexts, "scores": [score_by_id[chunk.chunk_id] for chunk in contexts]}


class HybridRRFPipeline:
    def __init__(self, *, retriever_model: str, generator_model: str, chunk_size: int, overlap: int, retrieve_k: int, top_k: int) -> None:
        self.dense = DenseRetriever(retriever_model)
        self.sparse = BM25Retriever()
        self.generator = SmallSeq2SeqGenerator(generator_model)
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.retrieve_k = retrieve_k
        self.top_k = top_k

    def index_document(self, record: dict[str, Any]) -> None:
        chunks = build_document_chunks(record, chunk_size=self.chunk_size, overlap=self.overlap)
        self.dense.index(chunks)
        self.sparse.index(chunks)

    def answer(self, question: str) -> dict[str, Any]:
        dense_results = self.dense.search(question, top_k=self.retrieve_k)
        sparse_results = self.sparse.search(question, top_k=self.retrieve_k)
        fused = reciprocal_rank_fusion([dense_results, sparse_results], top_k=self.top_k)
        contexts = [chunk for chunk, _score in fused]
        return {"answer": self.generator.answer(question, contexts), "contexts": contexts, "scores": [score for _chunk, score in fused]}


def build_pipeline(variant: str):
    if variant == "base_dense":
        return BaseDensePipeline(retriever_model=RETRIEVER_MODEL, generator_model=GENERATOR_MODEL, chunk_size=CHUNK_SIZE, overlap=OVERLAP, top_k=TOP_K)
    if variant == "bm25_only":
        return BM25OnlyPipeline(generator_model=GENERATOR_MODEL, chunk_size=CHUNK_SIZE, overlap=OVERLAP, top_k=TOP_K)
    if variant == "dense_u_shape":
        return DenseReorderPipeline(reorder_mode="u_shape", retriever_model=RETRIEVER_MODEL, generator_model=GENERATOR_MODEL, chunk_size=CHUNK_SIZE, overlap=OVERLAP, top_k=TOP_K)
    if variant == "dense_recency_heavy":
        return DenseReorderPipeline(reorder_mode="recency_heavy", retriever_model=RETRIEVER_MODEL, generator_model=GENERATOR_MODEL, chunk_size=CHUNK_SIZE, overlap=OVERLAP, top_k=TOP_K)
    if variant == "hybrid_rrf":
        return HybridRRFPipeline(retriever_model=RETRIEVER_MODEL, generator_model=GENERATOR_MODEL, chunk_size=CHUNK_SIZE, overlap=OVERLAP, retrieve_k=RETRIEVE_K, top_k=TOP_K)
    raise ValueError(f"Unknown variant: {variant}")

In [ ]:
def selected_records(dataset, *, min_doc_words: int):
    for record in dataset:
        if min_doc_words <= 0 or document_word_count(record) >= min_doc_words:
            yield record


def serialize_contexts(contexts: list[Chunk], scores: list[float]) -> list[dict[str, Any]]:
    return [
        {
            "chunk_id": chunk.chunk_id,
            "doc_id": chunk.doc_id,
            "title": chunk.title,
            "section": chunk.section,
            "text": chunk.text,
            "score": score,
        }
        for chunk, score in zip(contexts, scores)
    ]


def survey_dataset(dataset) -> dict[str, Any]:
    lengths = [document_word_count(record) for record in dataset]
    lengths_sorted = sorted(lengths)
    thresholds = [1000, 3000, 5000, 8000, 12000]
    return {
        "documents": len(lengths),
        "word_count_min": min(lengths_sorted),
        "word_count_median": int(median(lengths_sorted)),
        "word_count_mean": mean(lengths_sorted),
        "word_count_p90": lengths_sorted[round((len(lengths_sorted) - 1) * 0.90)],
        "word_count_max": max(lengths_sorted),
        "thresholds": {threshold: sum(1 for value in lengths if value >= threshold) for threshold in thresholds},
    }


def run_experiment(dataset) -> dict[str, Any]:
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    predictions_path = output_dir / f"{VARIANT}_{SPLIT}_min{MIN_DOC_WORDS}_predictions.jsonl"
    summary_path = output_dir / f"{VARIANT}_{SPLIT}_min{MIN_DOC_WORDS}_summary.json"

    pipeline = build_pipeline(VARIANT)
    totals = Counter()
    rows = 0
    docs_seen = 0
    start = time.perf_counter()

    with predictions_path.open("w", encoding="utf-8") as file:
        for record in tqdm(selected_records(dataset, min_doc_words=MIN_DOC_WORDS), desc=f"Running {VARIANT}"):
            docs_seen += 1
            pipeline.index_document(record)
            for example in extract_qa_examples(record):
                answer_result = pipeline.answer(example.question)
                contexts = answer_result["contexts"]
                scores = answer_result["scores"]
                prediction = answer_result["answer"]
                row_metrics = {
                    "token_f1": best_f1(prediction, example.gold_answers),
                    f"answer_string_recall_at_{TOP_K}": answer_string_recall(contexts, example.gold_answers),
                    "context_precision": context_precision(contexts, example.gold_answers, example.evidence),
                    "context_recall": context_recall(contexts, example.gold_answers, example.evidence),
                    "faithfulness": faithfulness(prediction, contexts),
                    "answer_relevancy": answer_relevancy(prediction, example.question, example.gold_answers),
                }
                row = {
                    "doc_id": example.doc_id,
                    "question_id": example.question_id,
                    "title": example.title,
                    "question": example.question,
                    "prediction": prediction,
                    "gold_answers": example.gold_answers,
                    "evidence": example.evidence,
                    "metrics": row_metrics,
                    "contexts": serialize_contexts(contexts, scores),
                }
                file.write(json.dumps(row, ensure_ascii=False) + "\n")
                totals.update(row_metrics)
                rows += 1
                if LIMIT is not None and rows >= LIMIT:
                    runtime = time.perf_counter() - start
                    metrics = {"examples": rows, **{f"avg_{key}": value / rows for key, value in totals.items()}}
                    summary = {
                        "variant": VARIANT,
                        "split": SPLIT,
                        "min_doc_words": MIN_DOC_WORDS,
                        "docs_seen": docs_seen,
                        "runtime_seconds": runtime,
                        "seconds_per_example": runtime / rows if rows else 0.0,
                        "config": CONFIG,
                        "metrics": metrics,
                        "predictions_path": str(predictions_path),
                    }
                    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
                    return summary

    runtime = time.perf_counter() - start
    metrics = {"examples": rows, **{f"avg_{key}": value / rows for key, value in totals.items()}} if rows else {"examples": 0}
    summary = {
        "variant": VARIANT,
        "split": SPLIT,
        "min_doc_words": MIN_DOC_WORDS,
        "docs_seen": docs_seen,
        "runtime_seconds": runtime,
        "seconds_per_example": runtime / rows if rows else 0.0,
        "config": CONFIG,
        "metrics": metrics,
        "predictions_path": str(predictions_path),
    }
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    return summary

In [ ]:
ds = load_qasper(SPLIT)
print("Dataset survey:")
print(json.dumps(survey_dataset(ds), indent=2))
print("\nRunning experiment:")
summary = run_experiment(ds)
summary